# youthxAI: Python + Colab
## PMW Internship 2026, AI-Based 3D Reconstruction Track

**Shahram Shafiq | FAST NUCES Islamabad**

This notebook has two parts:
1. **Python fundamentals**: the core concepts from the Kaggle Python course, applied to 3D reconstruction problems
2. **Applied exercises**: using numpy and matplotlib to work with point cloud data the way a real reconstruction pipeline does

This is the foundation for Week 3 when we reconstruct actual heritage footage.

---
## Part 1: Python Fundamentals
### 1.1 Variables and Data Types

In [ ]:
site_name = "Lahore Fort"
total_photos = 120
focal_len = 35.0
has_gps = True

print("Site:", site_name)
print("Photos taken:", total_photos)
print("Focal length (mm):", focal_len)
print("Has GPS:", has_gps)
print()
print("Types:", type(site_name), type(total_photos), type(focal_len), type(has_gps))

### 1.2 Functions

In [ ]:
def meters_to_feet(dist_m):
    return dist_m * 3.28084

def good_overlap(score_a, score_b, min_overlap=60):
    """Returns True if two photos share enough overlap for SfM feature matching."""
    diff = abs(score_a - score_b)
    return diff >= min_overlap

def clamp_val(v, lo, hi):
    if v < lo:
        return lo
    if v > hi:
        return hi
    return v

print("10 meters =", meters_to_feet(10), "feet")
print("Overlap check (80 vs 15):", good_overlap(80, 15))
print("Clamp 150 to [0, 100]:", clamp_val(150, 0, 100))
print("Clamp -5 to [0, 100]:", clamp_val(-5, 0, 100))

### 1.3 Booleans and Conditionals

In [ ]:
def pick_reconstruction_method(num_views, has_gpu):
    if num_views >= 50 and has_gpu:
        return "3D Gaussian Splatting (best quality + real-time)"
    elif num_views >= 20 and has_gpu:
        return "NeRF (photorealistic but slow)"
    elif num_views >= 10:
        return "COLMAP sparse reconstruction"
    else:
        return "Monocular depth (single image fallback)"

test_cases = [
    (80, True),
    (25, True),
    (12, False),
    (3, False),
]

print("Reconstruction method selector:")
for views, gpu in test_cases:
    result = pick_reconstruction_method(views, gpu)
    print(f"  {views} views, GPU={gpu}  ->  {result}")

### 1.4 Lists

In [ ]:
camera_spots = [
    [0.0, 1.5, -5.0],
    [2.0, 1.5, -4.5],
    [-2.0, 2.0, -4.0],
    [0.5, 3.0, -6.0],
]

print("Total cameras:", len(camera_spots))
print("First camera:", camera_spots[0])
print("Last camera:", camera_spots[-1])
print("First 2:", camera_spots[:2])

camera_spots.append([-1.0, 1.0, -3.5])
print("After adding one:", len(camera_spots), "cameras")

z_depths = []
for cam in camera_spots:
    z_depths.append(cam[2])
print("Z positions:", z_depths)

### 1.5 Loops

In [ ]:
import random
random.seed(42)

sharpness = [random.uniform(0.3, 1.0) for i in range(10)]

good_count = 0
bad_count = 0

print("Checking 10 photos for sharpness:")
for i in range(len(sharpness)):
    score = sharpness[i]
    if score >= 0.6:
        label = "OK"
        good_count += 1
    else:
        label = "BLURRY"
        bad_count += 1
    print(f"  Photo {i+1:02d}: score={score:.2f}  ->  {label}")

print(f"\nResult: {good_count} usable, {bad_count} rejected")

# While loop: how many shots needed to hit 95% coverage
coverage = 0.0
shots = 0
while coverage < 95.0:
    coverage += random.uniform(3.0, 8.0)
    shots += 1
    if coverage > 100.0:
        coverage = 100.0

print(f"Photos needed for 95% scene coverage: {shots}")

### 1.6 Strings and Dictionaries

In [ ]:
raw_fname = "  lahore_fort_shot_042.JPG  "
clean = raw_fname.strip().lower()
tokens = clean.split("_")

print("Original:", repr(raw_fname))
print("Cleaned:", clean)
print("Tokens:", tokens)
print("Is JPG:", clean.endswith(".jpg"))
print("Shot number:", tokens[-1].replace(".jpg", ""))

print()

heritage_db = {
    "Lahore Fort": {"city": "Lahore", "era": "Mughal", "photos": 340},
    "Rohtas Fort": {"city": "Jhelum", "era": "Sur Empire", "photos": 180},
    "Mohenjo-daro": {"city": "Larkana", "era": "Indus Valley", "photos": 95},
}

print("Heritage sites in database:")
for name in heritage_db:
    info = heritage_db[name]
    print(f"  {name} ({info['city']}, {info['era']}) - {info['photos']} photos")

heritage_db["Badshahi Mosque"] = {"city": "Lahore", "era": "Mughal", "photos": 220}
print("\nAfter adding Badshahi Mosque:", len(heritage_db), "sites total")

### 1.7 Working with External Libraries

In [ ]:
import math

def horizontal_fov(sensor_w_mm, focal_mm):
    angle = 2 * math.atan(sensor_w_mm / (2 * focal_mm))
    return math.degrees(angle)

sensor_w = 36.0  # full-frame sensor
print("Horizontal FOV for common lenses (full frame):")
for focal in [24, 35, 50, 85, 135]:
    fov = horizontal_fov(sensor_w, focal)
    print(f"  {focal}mm  ->  {fov:.1f} degrees")

print()

def dist_3d(a, b):
    dx = b[0] - a[0]
    dy = b[1] - a[1]
    dz = b[2] - a[2]
    return math.sqrt(dx*dx + dy*dy + dz*dz)

cam_a = [0.0, 1.5, -5.0]
cam_b = [2.0, 1.5, -4.5]
print(f"Baseline between cameras: {dist_3d(cam_a, cam_b):.4f} units")
print("(Wider baseline = better triangulation accuracy in SfM)")

---
## Part 2: Applied to 3D Reconstruction
### 2.1 NumPy for Point Cloud Data

In [ ]:
import numpy as np

np.random.seed(11)

# Synthetic point cloud: wall + ground of a heritage structure
wall_x = np.random.uniform(-3, 3, 200)
wall_y = np.random.uniform(0, 4, 200)
wall_z = np.ones(200) * 2.0 + np.random.normal(0, 0.05, 200)

ground_x = np.random.uniform(-3, 3, 150)
ground_y = np.zeros(150) + np.random.normal(0, 0.03, 150)
ground_z = np.random.uniform(0, 4, 150)

wall_cloud = np.column_stack([wall_x, wall_y, wall_z])
ground_cloud = np.column_stack([ground_x, ground_y, ground_z])
full_cloud = np.vstack([wall_cloud, ground_cloud])

print("Point cloud shape:", full_cloud.shape)
print(f"X range: {full_cloud[:,0].min():.2f} to {full_cloud[:,0].max():.2f}")
print(f"Y range: {full_cloud[:,1].min():.2f} to {full_cloud[:,1].max():.2f}")
print(f"Z range: {full_cloud[:,2].min():.2f} to {full_cloud[:,2].max():.2f}")

centroid = full_cloud.mean(axis=0)
centered_cloud = full_cloud - centroid
print(f"\nCentroid: {centroid.round(3)}")
print(f"After centering: {centered_cloud.mean(axis=0).round(8)}")
print("\nCentering puts the origin at the middle of the scene.")
print("This is standard preprocessing before NeRF or Gaussian Splatting training.")

### 2.2 Perspective Projection (Camera Model)

This is the core math that determines which 3D points each camera can see. SfM uses this to match feature points across images.

In [ ]:
import numpy as np

def project_to_camera(pts3d, cam_pos, focal=80.0, img_w=320, img_h=240):
    cx = img_w / 2.0
    cy = img_h / 2.0
    shifted = pts3d - cam_pos
    depth = shifted[:, 2]
    visible = depth > 0.5
    px = focal * shifted[visible, 0] / depth[visible] + cx
    py = focal * shifted[visible, 1] / depth[visible] + cy
    in_bounds = (px >= 0) & (px < img_w) & (py >= 0) & (py < img_h)
    return px[in_bounds], py[in_bounds], np.where(visible)[0][in_bounds]

np.random.seed(11)
scene_pts = np.random.uniform(-2, 2, (400, 3))
scene_pts[:, 2] = np.abs(scene_pts[:, 2]) + 1.0

cam_front = np.array([0.0, 0.0, -6.0])
cam_left  = np.array([-4.0, 0.0, -4.0])
cam_above = np.array([0.0, 4.0, -5.0])

px_f, py_f, idx_f = project_to_camera(scene_pts, cam_front)
px_l, py_l, idx_l = project_to_camera(scene_pts, cam_left)
px_a, py_a, idx_a = project_to_camera(scene_pts, cam_above)

overlap_fl = len(set(idx_f) & set(idx_l))

print(f"Front camera:  {len(px_f)} points visible")
print(f"Left camera:   {len(px_l)} points visible")
print(f"Above camera:  {len(px_a)} points visible")
print(f"\nFront-Left overlap: {overlap_fl} shared points")
print("These shared points are what COLMAP triangulates to get 3D positions.")

### 2.3 Visualization with Matplotlib

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

np.random.seed(11)

wall_pts = np.column_stack([
    np.random.uniform(-3, 3, 200),
    np.random.uniform(0, 4, 200),
    np.ones(200) * 2.0 + np.random.normal(0, 0.05, 200)
])
ground_pts = np.column_stack([
    np.random.uniform(-3, 3, 150),
    np.zeros(150) + np.random.normal(0, 0.03, 150),
    np.random.uniform(0, 4, 150)
])

cameras = {
    "Front":  np.array([0.0,  0.0, -6.0]),
    "Left":   np.array([-4.0, 0.0, -4.0]),
    "Top":    np.array([0.0,  4.0, -5.0]),
}

fig = plt.figure(figsize=(15, 5))

ax1 = fig.add_subplot(131, projection='3d')
ax1.scatter(wall_pts[:,0], wall_pts[:,2], wall_pts[:,1], c='steelblue', s=5, alpha=0.6, label='Wall')
ax1.scatter(ground_pts[:,0], ground_pts[:,2], ground_pts[:,1], c='sienna', s=5, alpha=0.6, label='Ground')
for cname, cpos in cameras.items():
    ax1.scatter(cpos[0], cpos[2], cpos[1], c='red', s=60, marker='^', zorder=5)
    ax1.text(cpos[0], cpos[2], cpos[1]+0.2, cname, fontsize=7, color='red')
ax1.set_title('3D Point Cloud + Cameras', fontsize=9, fontweight='bold')
ax1.set_xlabel('X')
ax1.set_ylabel('Z (depth)')
ax1.set_zlabel('Y (height)')
ax1.legend(fontsize=7)

ax2 = fig.add_subplot(132)
all_pts = np.vstack([wall_pts, ground_pts])
cols = ['steelblue'] * 200 + ['sienna'] * 150
ax2.scatter(all_pts[:,0], all_pts[:,2], c=cols, s=6, alpha=0.7)
for cname, cpos in cameras.items():
    ax2.scatter(cpos[0], cpos[2], c='red', s=80, marker='^', zorder=5)
    ax2.annotate(cname, (cpos[0], cpos[2]), fontsize=7, color='red', ha='center')
ax2.set_title('Top-Down View (Floor Plan)', fontsize=9, fontweight='bold')
ax2.set_xlabel('X')
ax2.set_ylabel('Z (depth)')
ax2.set_aspect('equal')
ax2.grid(True, alpha=0.3)

ax3 = fig.add_subplot(133)
ax3.scatter(wall_pts[:,0], wall_pts[:,1], c='steelblue', s=5, alpha=0.6, label='Wall')
ax3.scatter(ground_pts[:,0], ground_pts[:,1], c='sienna', s=5, alpha=0.6, label='Ground')
ax3.set_title('Front View (Elevation)', fontsize=9, fontweight='bold')
ax3.set_xlabel('X')
ax3.set_ylabel('Y (height)')
ax3.legend(fontsize=7)
ax3.grid(True, alpha=0.3)

plt.suptitle('Heritage Site Point Cloud | PMW 2026 | Shahram Shafiq', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('point_cloud_viz.png', dpi=130, bbox_inches='tight')
plt.show()
print("Saved: point_cloud_viz.png")

---
## Part 3: Kaggle Python Course Summary

Completed all 7 modules of the [Kaggle Python course](https://www.kaggle.com/learn/python):

| # | Module | Key concept | Used above |
|---|---|---|---|
| 1 | Hello, Python | Syntax, variables, arithmetic | Section 1.1 |
| 2 | Functions and Getting Help | `def`, docstrings, `help()` | Section 1.2 |
| 3 | Booleans and Conditionals | `if/elif/else`, `and/or` | Section 1.3 |
| 4 | Lists | Indexing, slicing, `.append()` | Section 1.4 |
| 5 | Loops and List Comprehensions | `for`, `while`, iteration | Section 1.5 |
| 6 | Strings and Dictionaries | String methods, dict operations | Section 1.6 |
| 7 | Working with External Libraries | `import`, `math`, `numpy` | Sections 1.7, 2.1-2.3 |

Every concept from the Kaggle course shows up directly in the 3D reconstruction code in Part 2.

---
## Summary

**What this notebook covers:**
- All 7 Kaggle Python course topics applied to 3D reconstruction problems
- NumPy arrays for point cloud data (the output format of COLMAP/SfM)
- Perspective projection: the math that maps 3D scene points into 2D camera images
- Matplotlib: three views of a synthetic heritage building point cloud (3D, top-down, elevation)

**Connection to the PMW pipeline:**
- Section 2.1 = the data structure SfM produces (N x 3 point array)
- Section 2.2 = the math COLMAP uses internally to find camera positions
- Section 2.3 = how we inspect reconstruction quality before running Gaussian Splatting

**Week 3 goal:** apply these tools to real footage of a heritage site and produce an actual 3D reconstruction using COLMAP.